# NL → SQL with T5-small
Fine-tune a T5-small model on the [WikiSQL](https://github.com/salesforce/WikiSQL) dataset to translate natural language questions into SQL queries.

**Sections:**
1. Install dependencies
2. Configuration
3. Download & extract dataset
4. Preprocess — convert WikiSQL → plain SQL
5. Build training pairs
6. Tokenize with T5
7. Fine-tune the model
8. Interactive inference

## 1 · Install dependencies

In [2]:
%pip install -q transformers torch sentencepiece sqlparse requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.1 MB/s eta 0:00:00


## 2 · Configuration
All hyperparameters in one place — edit here and every cell below picks up the change.

In [3]:
# ── Model & tokenizer ─────────────────────────────────────────────────────────
MODEL_NAME         = "t5-small"
MAX_INPUT_LENGTH   = 256
MAX_OUTPUT_LENGTH  = 128

# ── Training ──────────────────────────────────────────────────────────────────
LEARNING_RATE      = 3e-4
BATCH_SIZE         = 16
NUM_EPOCHS         = 2
SAVE_STEPS         = 50
LOGGING_STEPS      = 50
MAX_TRAIN_SAMPLES  = 8000

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR           = "data"
MODEL_OUTPUT_DIR   = "./sql_model"
DATASET_URL        = "https://github.com/salesforce/WikiSQL/raw/master/data.tar.bz2"
ARCHIVE_NAME       = "data.tar.bz2"

# ── Inference ─────────────────────────────────────────────────────────────────
NUM_BEAMS              = 4
NO_REPEAT_NGRAM_SIZE   = 3

print("Config ready.")

Config ready.


## 3 · Download & extract the WikiSQL dataset

In [4]:
import tarfile
import requests

def download_and_extract() -> None:
    print(f"Downloading dataset from {DATASET_URL} ...")
    response = requests.get(DATASET_URL, stream=True)
    total = int(response.headers.get("content-length", 0))

    with open(ARCHIVE_NAME, "wb") as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\rProgress: {downloaded}/{total} bytes ({100*downloaded//total}%)", end="")

    print("\nDownload complete.")
    print("Extracting archive...")
    with tarfile.open(ARCHIVE_NAME, "r:bz2") as tar:
        tar.extractall()
    print("Extraction complete.")

download_and_extract()

Progress: 26164664/26164664 bytes (100%)
Download complete.
Extracting archive...


/tmp/ipykernel_7690/1957603183.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction complete.


## 4 · Preprocess — convert WikiSQL format → plain SQL

In [5]:
import json

AGG_OPS  = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<", "OP"]

def convert_sql(sample: dict, table: dict) -> str:
    """Convert a WikiSQL sample + table dict into a plain SQL string."""
    col_names   = table["header"]
    select_col  = col_names[sample["sql"]["sel"]]
    agg         = AGG_OPS[sample["sql"]["agg"]]
    select_part = f"{agg}({select_col})" if agg else select_col

    where_clauses = []
    for col_idx, op_idx, value in sample["sql"]["conds"]:
        where_clauses.append(f"{col_names[col_idx]}{COND_OPS[op_idx]}'{value}'")

    query = f"SELECT {select_part} FROM table"
    if where_clauses:
        query += " WHERE " + " AND ".join(where_clauses)
    return query

def build_schema(table: dict) -> str:
    """Return a compact schema string, e.g. 'name(text), age(real)'."""
    return ", ".join(
        f"{col}({typ})" for col, typ in zip(table["header"], table["types"])
    )

# Quick sanity check
with open(f"{DATA_DIR}/train.jsonl")        as f: sample = json.loads(next(f))
with open(f"{DATA_DIR}/train.tables.jsonl") as f: table  = json.loads(next(f))

print("Question :", sample["question"])
print("Schema   :", build_schema(table))
print("SQL      :", convert_sql(sample, table))

Question : Tell me what the notes are for South Australia 
Schema   : State/territory(text), Text/background colour(text), Format(text), Current slogan(text), Current series(text), Notes(text)
SQL      : SELECT Notes FROM table WHERE Current slogan='SOUTH AUSTRALIA'


## 5 · Build training pairs

In [6]:
from typing import List, Tuple

def create_training_examples() -> List[Tuple[str, str]]:
    tables: dict = {}
    with open(f"{DATA_DIR}/train.tables.jsonl") as f:
        for line in f:
            t = json.loads(line)
            tables[t["id"]] = t

    pairs = []
    with open(f"{DATA_DIR}/train.jsonl") as f:
        for line in f:
            s      = json.loads(line)
            table  = tables[s["table_id"]]
            inp    = f"Schema: {build_schema(table)}, Question: {s['question']}"
            sql    = convert_sql(s, table)
            pairs.append((inp, sql))

    return pairs[:MAX_TRAIN_SAMPLES]

training_pairs = create_training_examples()
print(f"{len(training_pairs)} training pairs built.")
print("\nSample input :", training_pairs[0][0])
print("Sample target:", training_pairs[0][1])

8000 training pairs built.

Sample input : Schema: State/territory(text), Text/background colour(text), Format(text), Current slogan(text), Current series(text), Notes(text), Question: Tell me what the notes are for South Australia 
Sample target: SELECT Notes FROM table WHERE Current slogan='SOUTH AUSTRALIA'


## 6 · Tokenize with T5

In [7]:
from torch.utils.data import Dataset
from transformers import T5Tokenizer

class SQLDataset(Dataset):
    """Tokenizes (input, target) string pairs for T5 fine-tuning."""

    def __init__(self, pairs, tokenizer):
        self.pairs     = pairs
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        inp, out = self.pairs[idx]
        tokens = self.tokenizer(
            inp,
            text_target=out,
            truncation=True,
            padding="max_length",
            max_length=MAX_INPUT_LENGTH,
        )
        return {
            "input_ids":      tokens["input_ids"],
            "attention_mask": tokens["attention_mask"],
            "labels":         tokens["labels"],
        }

tokenizer     = T5Tokenizer.from_pretrained(MODEL_NAME)
train_dataset = SQLDataset(training_pairs, tokenizer)

print(f"Dataset size: {len(train_dataset)}")
print("Sample keys :", list(train_dataset[0].keys()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset size: 8000
Sample keys : ['input_ids', 'attention_mask', 'labels']


## 7 · Fine-tune the model
> ⏱ Expected time: ~15–30 min on CPU, ~3–5 min on GPU.

In [8]:
import time
from transformers import (
    T5ForConditionalGeneration,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

class TimingCallback(TrainerCallback):
    def __init__(self):
        self.train_start = self.epoch_start = 0.0

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        print(f"\n{'='*55}\n  Training started — Total steps: {state.max_steps}\n{'='*55}\n")

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start = time.time()
        print(f"\n── Epoch {int(state.epoch or 0) + 1} started ──")

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0 and state.global_step > 0:
            elapsed   = time.time() - self.train_start
            steps_left = state.max_steps - state.global_step
            eta_sec   = (elapsed / state.global_step) * steps_left
            print(
                f"  Step {state.global_step:>4}/{state.max_steps} "
                f"({100*state.global_step/state.max_steps:5.1f}%) | "
                f"Elapsed: {time.strftime('%H:%M:%S', time.gmtime(elapsed))} | "
                f"ETA: {time.strftime('%H:%M:%S', time.gmtime(eta_sec))}"
            )

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n── Epoch {int(state.epoch or 0)} done in "
              f"{time.time()-self.epoch_start:.1f}s ──\n")

    def on_train_end(self, args, state, control, **kwargs):
        total = time.time() - self.train_start
        print(f"\n{'='*55}\n  Training complete!  "
              f"Total time: {time.strftime('%H:%M:%S', time.gmtime(total))}\n{'='*55}\n")


model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    disable_tqdm=True,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[TimingCallback()],
)

trainer.train()
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Model saved to '{MODEL_OUTPUT_DIR}'.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


  Training started — Total steps: 1000


── Epoch 1 started ──


...

  Step  960/1000 ( 96.0%) | Elapsed: 00:06:37 | ETA: 00:00:16
  Step  970/1000 ( 97.0%) | Elapsed: 00:06:39 | ETA: 00:00:12
  Step  980/1000 ( 98.0%) | Elapsed: 00:06:40 | ETA: 00:00:08
  Step  990/1000 ( 99.0%) | Elapsed: 00:06:42 | ETA: 00:00:04
  Step 1000/1000 (100.0%) | Elapsed: 00:06:43 | ETA: 00:00:00
{'loss': '0.02517', 'grad_norm': '0.1369', 'learning_rate': '3e-07', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


── Epoch 2 done in 166.5s ──

{'train_runtime': '410.7', 'train_samples_per_second': '38.96', 'train_steps_per_second': '2.435', 'train_loss': '0.07996', 'epoch': '2'}

  Training complete!  Total time: 00:06:50



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to './sql_model'.


## 8 · Interactive inference
The cell below loads the saved model and lets you query it. Re-run it as many times as you like without retraining.

In [9]:
import torch
import sqlparse
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load saved model
tokenizer = T5Tokenizer.from_pretrained(MODEL_OUTPUT_DIR)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_OUTPUT_DIR)
device    = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print(f"Model loaded on {device}.")

def generate_sql(schema: str, question: str) -> str:
    input_text = f"Schema: {schema}, Question: {question}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            num_beams=NUM_BEAMS,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            early_stopping=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def is_valid_sql(sql: str) -> bool:
    return bool(sqlparse.parse(sql.strip()))

# ── Try it out ────────────────────────────────────────────────────────────────
schema   = "name(text), age(real), salary(real)"
question = "What is the average salary of employees older than 30?"

sql = generate_sql(schema, question)
print("Generated SQL:", sql)
print("Valid SQL?    ", is_valid_sql(sql))

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on cpu.
Generated SQL: SELECT MAX(Salary) FROM table WHERE age='30'
Valid SQL?     True


In [10]:
# ── Interactive loop — keep running this cell with different inputs ────────────
schema   = input("Table schema (e.g. 'name(text), age(real), salary(real)'): ").strip()
question = input("Question: ").strip()

sql = generate_sql(schema, question)
print("\nGenerated SQL:", sql)
if not is_valid_sql(sql):
    print("⚠ Warning: the generated query may not be valid SQL.")

Table schema (e.g. 'name(text), age(real), salary(real)'): Name(text), Age(number), Salary(number), Department(text)
Question: How many records are there in Sales Department?

Generated SQL: SELECT COUNT(Name) FROM table WHERE Department='Sales Department'
